In [ ]:
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import AutoModelForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import QuantizationModifier
from compressed_tensors.offload import dispatch_model

In [ ]:
MODEL_ID = "zai-org/GLM-5.1"
SAVE_DIR = "GLM-5.1-NVFP4"
NUM_CALIBRATION_SAMPLES = 512
MAX_SEQUENCE_LENGTH = 2048


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map=None,
    cache_dir="/workspace",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir="/workspace")

config.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Cancellation requested; stopping current tasks.


In [ ]:
# ds = load_dataset("HuggingFaceH4/ultrachat_200k", split=f"train_sft[:{NUM_CALIBRATION_SAMPLES}]")
# ds = ds.shuffle(seed=42)

# https://huggingface.co/datasets/tejeshbhalla/tool_calling/viewer/default/train?row=1
# https://huggingface.co/datasets/AymanTarig/function-calling-v0.2-with-r1-cot

code = load_dataset("nvidia/Nemotron-Post-Training-Dataset-v2", "SFT", split="code")
math = load_dataset("nvidia/Nemotron-Post-Training-Dataset-v2", "SFT", split="math")
ds = concatenate_datasets([code, math])
ds = ds.shuffle(seed=1337).select(range(NUM_CALIBRATION_SAMPLES))

def preprocess(example):
    return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

ds = ds.map(preprocess)

def tokenize(sample):
    return tokenizer(
        sample["text"],
        padding=False,
        max_length=MAX_SEQUENCE_LENGTH,
        truncation=True,
        add_special_tokens=False,
    )

ds = ds.map(tokenize, remove_columns=ds.column_names)

In [ ]:
recipe = QuantizationModifier(
    targets="Linear",
    scheme="NVFP4",
    ignore=[
        "re:.*lm_head",
        "re:.*mlp.gate$",
        "re:.*embed_tokens$",
        "re:.*shared_expert_gate$",
    ],
)

In [ ]:
oneshot(
    model=model,
    recipe=recipe,
    dataset=ds,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
    moe_calibrate_all_experts=True,
)

In [ ]:
print("========== SAMPLE GENERATION ==============")
dispatch_model(model)
input_ids = tokenizer("Hello my name is", return_tensors="pt").input_ids.to(model.device)
output = model.generate(input_ids, max_new_tokens=20)
print(tokenizer.decode(output[0]))
print("==========================================")

# %%
model.save_pretrained(SAVE_DIR, save_compressed=True)
tokenizer.save_pretrained(SAVE_DIR)